# Complementos de Investigação Operacional — capítulo 6: Filas de espera

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/applied-operations-research-course/blob/main/code/python/06_queueing/06_1_mm1/ex06_1_mm1.ipynb)

Corra a célula **Preparação** e depois o exemplo (no Colab: *Runtime → Run all*). No fim, o exemplo compara os resultados com os slides (*confere com os slides: sim*). Os slides usam vírgula decimal; aqui usa-se o ponto. Caderno gerado a partir de `ex06_1_mm1.py` por `notebooks/make_notebooks.py`.

*Code comments and printed messages are in Portuguese.*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("applied-operations-research-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/applied-operations-research-course
    RAIZ = os.path.abspath(os.path.join("applied-operations-research-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 6.1 Filas de espera: conceitos, lei de Little e M/M/1

Exemplo [`ex06_1_mm1.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/06_queueing/06_1_mm1/ex06_1_mm1.py) · funções da UC: [`filas.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/06_queueing/06_1_mm1/filas.py)

```text
Reproduz os exemplos do deck 6.1 (conceitos, lei de Little e M/M/1).

Exemplo-guia: posto de carregamento rápido com um carregador, lambda = 3 carros/h, mu = 4/h:
    rho = 0.75, P0 = 0.25, L = 3, Lq = 2.25, W = 1 h, Wq = 45 min; Pn = 0.25*0.75^n;
    P(n>=1) = 75%, P(n>=3) = 42%, P(n>=8) = 10%; Little na loja (40/120 h = 20 min).
Efeito da utilização: lambda = 2, 2.5, 3, 3.5, 3.8 -> Wq = 15, 25, 45, 105, 285 min.
Cauda: 95% esperam menos de 2.71 h; a simulação (simula_mms(3, 4, 1), o código do slide)
dá 45.5 min, 0.75 e 2.72 h e compara-se com as fórmulas com tolerância.
Para resolver na aula: Exemplo 1 (controlo de bagagens, lambda = 10/min, mu = 12/min):
83.3%, 69.4%, 4.17 passageiros, 30 s.
```

In [2]:
import math

import numpy as np

from filas import mm1, simula_mms

ok = True
print("Complementos de IO — deck 6.1: filas de espera, lei de Little e M/M/1")

# ------------------------------------------------------------ posto de carregamento
lam, mu = 3, 4
m = mm1(lam, mu)
print("\nPosto de carregamento rápido (M/M/1, lambda = %g/h, mu = %g/h)" % (lam, mu))
print("  rho = %.4g  P0 = %.4g  L = %.4g  Lq = %.4g  W = %.4g h  Wq = %.4g min"
      % (m['rho'], m['P0'], m['L'], m['Lq'], m['W'], m['Wq'] * 60))
print("  Little: lambda*W = %.4g (= L);  lambda*Wq = %.4g (= Lq)" % (lam * m['W'], lam * m['Wq']))
Pn = np.array([m['Pn'](n) for n in range(9)])
print("  Pn, n = 0..8: %s" % " ".join("%.4f" % x for x in Pn))
pmaior = [m['rho'] ** k for k in (1, 3, 8)]          # P(n >= k) = rho^k
print("  P(n>=1) = %.0f%%  P(n>=3) = %.0f%%  P(n>=8) = %.0f%%" % tuple(100 * p for p in pmaior))
print("Little na loja: 40 clientes, 120 entradas/h -> %.4g min por cliente" % (40 / 120 * 60))
ok &= (np.allclose([m['rho'], m['P0'], m['L'], m['Lq'], m['W'], m['Wq'] * 60], [0.75, 0.25, 3, 2.25, 1, 45])
       and np.allclose(np.round(Pn, 4), [0.25, 0.1875, 0.1406, 0.1055, 0.0791, 0.0593, 0.0445, 0.0334, 0.025])
       and np.allclose(np.round(pmaior, 2), [0.75, 0.42, 0.10]))

# ------------------------------------------------------------ efeito da utilização
print("\nO efeito da utilização (mu = 4/h)")
Wq = []
for l in (2, 2.5, 3, 3.5, 3.8):
    x = mm1(l, mu)
    Wq.append(x['Wq'] * 60)
    print("  lambda = %.1f/h  rho = %.3f  Wq = %.4g min" % (l, x['rho'], Wq[-1]))
print("  3 -> 3.5 carros/h (+%.0f%%): Wq passa de %.4g para %.4g min (x%.2f)"
      % (100 * 0.5 / 3, Wq[2], Wq[3], Wq[3] / Wq[2]))
ok &= np.allclose(Wq, [15, 25, 45, 105, 285])

# ------------------------------------------------------------ cauda e simulação
t95 = math.log(m['rho'] / 0.05) / (mu - lam)          # rho*exp(-(mu - lam) t) = 0.05
print("\nCauda: P(Wq > t) = rho*exp(-(mu - lambda)*t); 95%% esperam menos de %.2f h" % t95)
sim = simula_mms(lam, mu, 1)
print("Simulação (simula_mms(3, 4, 1): 200000 carros, semente 1, sem os 10000 primeiros)")
print("  Wq = %.1f min (fórmula %.4g)  P(esperar) = %.2f (%.2f)  quantil 95%% = %.2f h (%.2f)"
      % (sim['Wq'] * 60, m['Wq'] * 60, sim['Pw'], m['rho'], sim['p95'], t95))
tol = (abs(sim['Wq'] - m['Wq']) <= 0.10 * m['Wq'] and abs(sim['Pw'] - m['rho']) <= 0.02
       and abs(sim['p95'] - t95) <= 0.10 * t95)
print("  simulação dentro da tolerância (10%% nas médias e no quantil, 0.02 nas probabilidades): %s"
      % ("sim" if tol else "não"))
ok &= round(t95, 2) == 2.71 and tol

# ------------------------------------------------------------ Exemplo 1 (para resolver na aula)
b = mm1(10, 12)
print("\nExemplo 1 — controlo de bagagens (lambda = 10/min, mu = 12/min)")
print("  a) P(máquina não ociosa) = rho = %.4f (%.1f%%)" % (b['rho'], 100 * b['rho']))
print("  b) P(haver fila) = P(n>=2) = rho^2 = %.4f (%.1f%%)" % (b['rho'] ** 2, 100 * b['rho'] ** 2))
print("  c) Lq = %.4f passageiros" % b['Lq'])
print("  d) W = %.4g min = %.4g s (Wq = %.4g s mais o controlo)" % (b['W'], b['W'] * 60, b['Wq'] * 60))
ok &= (round(100 * b['rho'], 1) == 83.3 and round(100 * b['rho'] ** 2, 1) == 69.4
       and abs(b['Lq'] - 25 / 6) < 1e-9 and abs(b['W'] * 60 - 30) < 1e-9)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 6.1: filas de espera, lei de Little e M/M/1

Posto de carregamento rápido (M/M/1, lambda = 3/h, mu = 4/h)
  rho = 0.75  P0 = 0.25  L = 3  Lq = 2.25  W = 1 h  Wq = 45 min
  Little: lambda*W = 3 (= L);  lambda*Wq = 2.25 (= Lq)
  Pn, n = 0..8: 0.2500 0.1875 0.1406 0.1055 0.0791 0.0593 0.0445 0.0334 0.0250
  P(n>=1) = 75%  P(n>=3) = 42%  P(n>=8) = 10%
Little na loja: 40 clientes, 120 entradas/h -> 20 min por cliente

O efeito da utilização (mu = 4/h)
  lambda = 2.0/h  rho = 0.500  Wq = 15 min
  lambda = 2.5/h  rho = 0.625  Wq = 25 min
  lambda = 3.0/h  rho = 0.750  Wq = 45 min
  lambda = 3.5/h  rho = 0.875  Wq = 105 min
  lambda = 3.8/h  rho = 0.950  Wq = 285 min
  3 -> 3.5 carros/h (+17%): Wq passa de 45 para 105 min (x2.33)

Cauda: P(Wq > t) = rho*exp(-(mu - lambda)*t); 95% esperam menos de 2.71 h


Simulação (simula_mms(3, 4, 1): 200000 carros, semente 1, sem os 10000 primeiros)
  Wq = 45.5 min (fórmula 45)  P(esperar) = 0.75 (0.75)  quantil 95% = 2.72 h (2.71)
  simulação dentro da tolerância (10% nas médias e no quantil, 0.02 nas probabilidades): sim

Exemplo 1 — controlo de bagagens (lambda = 10/min, mu = 12/min)
  a) P(máquina não ociosa) = rho = 0.8333 (83.3%)
  b) P(haver fila) = P(n>=2) = rho^2 = 0.6944 (69.4%)
  c) Lq = 4.1667 passageiros
  d) W = 0.5 min = 30 s (Wq = 25 s mais o controlo)

confere com os slides: sim
